# ДЗ 3. Динамическое программирование
**Выполнил:** Хритинин Д. А.

## Climbing Stairs (обязательная)
Лестница из `n` ступенек, за шаг можно подняться на 1 или 2 ступеньки. Сколько существует способов добраться до вершины?

**Идея:** `dp[i]` — число способов дойти до ступеньки `i`. На ступеньку `i` можно прийти с `i-1` или с `i-2`, поэтому
`dp[i] = dp[i-1] + dp[i-2]`, база `dp[0] = dp[1] = 1` (это числа Фибоначчи со сдвигом).

| Способ | Время | Память |
|---|---|---|
| Мемоизация | O(n) | O(n) — кэш + стек рекурсии |
| Табуляция | O(n) | O(n) — массив dp |
| Две переменные | O(n) | **O(1)** |

In [ ]:
from functools import lru_cache

# Способ 1. Мемоизация (рекурсия сверху вниз + кэш)
def climb_stairs_memo(n: int) -> int:
    memo = {}
    def dp(i):
        if i <= 1:          # база: dp[0] = dp[1] = 1
            return 1
        if i not in memo:
            memo[i] = dp(i - 1) + dp(i - 2)
        return memo[i]
    return dp(n)


# Способ 2. Табуляция (снизу вверх, массив dp)
def climb_stairs_tab(n: int) -> int:
    if n <= 1:
        return 1
    dp = [0] * (n + 1)
    dp[0], dp[1] = 1, 1
    for i in range(2, n + 1):
        dp[i] = dp[i - 1] + dp[i - 2]
    return dp[n]


# Способ 3. O(1) по памяти — храним только два последних значения
def climb_stairs_opt(n: int) -> int:
    prev2, prev1 = 1, 1      # dp[i-2], dp[i-1]
    for _ in range(2, n + 1):
        prev2, prev1 = prev1, prev1 + prev2
    return prev1

In [ ]:
tests = {1: 1, 2: 2, 3: 3, 4: 5, 5: 8, 10: 89, 45: 1836311903}
for f in (climb_stairs_memo, climb_stairs_tab, climb_stairs_opt):
    for n, expected in tests.items():
        assert f(n) == expected, (f.__name__, n, f(n))
    print(f"{f.__name__}: все тесты пройдены")

print("n = 2 ->", climb_stairs_opt(2))
print("n = 3 ->", climb_stairs_opt(3))

## House Robber (*)
Нельзя грабить два соседних дома. Найти максимальную сумму.

**Идея:** `dp[i]` — максимум, который можно унести из первых `i+1` домов. Для дома `i` два варианта:
- пропустить его → `dp[i-1]`;
- ограбить → `dp[i-2] + nums[i]`.

`dp[i] = max(dp[i-1], dp[i-2] + nums[i])`, база `dp[0] = nums[0]`, `dp[1] = max(nums[0], nums[1])`.
Храним только две последние величины → **O(n) по времени, O(1) по памяти**.

Краевые случаи: пустой массив → 0, один дом → `nums[0]`, два дома → максимум из двух.

In [ ]:
def rob(nums: list[int]) -> int:
    # Краевые случаи
    if not nums:
        return 0
    if len(nums) == 1:
        return nums[0]
    # prev2 = dp[i-2], prev1 = dp[i-1]
    prev2, prev1 = nums[0], max(nums[0], nums[1])
    for i in range(2, len(nums)):
        prev2, prev1 = prev1, max(prev1, prev2 + nums[i])
    return prev1

In [ ]:
assert rob([1, 2, 3, 1]) == 4
assert rob([2, 7, 9, 3, 1]) == 12
assert rob([]) == 0          # пустой массив
assert rob([5]) == 5         # один дом
assert rob([2, 9]) == 9      # два дома
assert rob([2, 1, 1, 2]) == 4
print("rob: все тесты пройдены")
print("[1,2,3,1]   ->", rob([1, 2, 3, 1]))
print("[2,7,9,3,1] ->", rob([2, 7, 9, 3, 1]))

### ** Дома стоят по кругу
Теперь первый и последний дом — соседи, и ограбить оба нельзя. Значит, хотя бы один из них точно не грабим.
Рассматриваем два случая: улица без последнего дома (`nums[:-1]`) и улица без первого дома (`nums[1:]`).
Каждый случай — обычная линейная задача, решаем её функцией `rob` и берём максимум.
Сложность остаётся O(n) по времени и O(1) по дополнительной памяти (если передавать индексы вместо срезов).

In [ ]:
def rob_circle(nums: list[int]) -> int:
    # Дома по кругу: первый и последний — соседи, их нельзя брать вместе.
    # Значит, решаем линейную задачу дважды:
    #   1) без последнего дома  nums[:-1]
    #   2) без первого дома     nums[1:]
    # и берём максимум.
    if not nums:
        return 0
    if len(nums) == 1:
        return nums[0]
    return max(rob(nums[:-1]), rob(nums[1:]))


assert rob_circle([2, 3, 2]) == 3
assert rob_circle([1, 2, 3, 1]) == 4
assert rob_circle([1, 2, 3]) == 3
assert rob_circle([]) == 0
assert rob_circle([7]) == 7
print("rob_circle: все тесты пройдены")